In [ ]:
#Imports 
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import OneHotEncoder
from ID3 import ID3Classifier
from sklearn.metrics import f1_score

ruta = "futbol_uruguayo.csv"
ds = pd.read_csv(ruta)

In [ ]:
#Preprocesamiento
#-1 Eligo los atributos del dataset
atributos = ["home_ident", "away_ident", "historial", "nivel_relativo"]

#0. Genero un split
tscv = TimeSeriesSplit(n_splits=3) #Con kfoldin habria data leakage 

#1. Calculo columna resultado
ds["resultado"] = np.select(
    [ds["gh"] > ds["ga"], ds["gh"] < ds["ga"]], ["G", "P"], default="E"
)
ds["date"] = pd.to_datetime(ds["date"])

#2. Creacion de nuevas columnas de datos: historial, nivelRelativo 
def calcular_historial_y_nivel(ds):
    # Ordenamos cronológicamente
    ds = ds.sort_values("date").copy()

    # Nuevas columnas
    ds["historial"] = "Neutro"
    ds["nivel_relativo"] = "Parejo"

    # Recorremos cada partido
    for i, fila in ds.iterrows():

        local = fila["home_ident"]
        visitante = fila["away_ident"]
        fecha = fila["date"]

        # -----------------------------------------
        # Buscar enfrentamientos anteriores
        # entre estos dos equipos
        # -----------------------------------------

        anteriores = ds.loc[
            (ds["date"] < fecha) &
            (
                ((ds["home_ident"] == local) & (ds["away_ident"] == visitante)) |
                ((ds["home_ident"] == visitante) & (ds["away_ident"] == local))
            )
        ].tail(10)

        # Si no hay enfrentamientos anteriores
        if len(anteriores) == 0:
            continue

        victorias = 0
        derrotas = 0
        goles_favor = []
        goles_contra = []

        # -----------------------------------------
        # Analizamos los últimos enfrentamientos
        # desde la perspectiva del equipo LOCAL
        # de la fila actual
        # -----------------------------------------

        for _, partido in anteriores.iterrows():

            if (
                partido["home_ident"] == local
                and partido["away_ident"] == visitante
            ):
                gf = partido["gh"]
                gc = partido["ga"]
            else:
                # El equipo actual jugó de visitante
                gf = partido["ga"]
                gc = partido["gh"]

            goles_favor.append(gf)
            goles_contra.append(gc)

            if gf > gc:
                victorias += 1
            elif gf < gc:
                derrotas += 1

        # -----------------------------------------
        # 1. Historial
        # -----------------------------------------

        if victorias > derrotas:
            ds.at[i, "historial"] = "Positivo"
        elif derrotas > victorias:
            ds.at[i, "historial"] = "Negativo"
        else:
            ds.at[i, "historial"] = "Neutro"

        # -----------------------------------------
        # 2. Nivel relativo (diferencia de forma reciente)
        # -----------------------------------------

        promedio_gf = sum(goles_favor) / len(goles_favor)
        promedio_gc = sum(goles_contra) / len(goles_contra)
        diferencia = promedio_gf - promedio_gc

        if diferencia <= -1.5:
            categoria_nivel = "Muy_Inferior"
        elif diferencia <= -0.5:
            categoria_nivel = "Inferior"
        elif diferencia < 0.5:
            categoria_nivel = "Parejo"
        elif diferencia < 1.5:
            categoria_nivel = "Superior"
        else:
            categoria_nivel = "Muy_Superior"

        ds.at[i, "nivel_relativo"] = categoria_nivel

    return ds
ds = calcular_historial_y_nivel(ds)

#3. Separacion en datos de Entrenamiento y Test
mask_test = ds["date"].dt.year >= 2024
train = ds[~mask_test]
test = ds[mask_test]

X_train = train[atributos] 
y_train = train["resultado"]

X_test = test[atributos]
y_test = test["resultado"]


#4. Aplico pipeline para el resto del preprocessing
atributes_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ))
])

preprocessing = ColumnTransformer([
    ('atributes', atributes_pipeline, atributos)
])

In [ ]:
#Estimador 1 (sklearn - Random Forest)
pipeline_rf = Pipeline([
    ('preprocessing', preprocessing),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    "model__n_estimators": [10, 25, 50, 75, 100, 150],
    "model__max_depth": [1, 3, 5, 7, None],
}

random_rf = RandomizedSearchCV(
    estimator= pipeline_rf,
    param_distributions=param_grid_rf,
    cv=tscv,
    scoring="f1_macro",
    n_jobs=-1,
    n_iter=24,
    random_state=42
)

random_rf.fit(X_train, y_train)

y_pred_rf = random_rf.predict(X_test)

In [ ]:
#Estimador 2 (sklearn - Naive Bayes)
pipeline_nb = Pipeline([
    ('preprocessing', preprocessing),
    ('model', CategoricalNB())
])

# Suavizado de Laplace/Lidstone. Ampliamos el rango en escala log: con valores
# chicos (0.01-2, lo que habia antes) el macro-F1 en CV queda practicamente
# plano, el efecto recien se nota a partir de alpha~10 y se derrumba para
# alpha grandes (el suavizado termina dominando sobre los datos).
param_grid_nb = {"model__alpha": [0.01, 0.1, 1, 5, 10, 50, 100, 500, 1000]}

random_nb = RandomizedSearchCV(
    estimator=pipeline_nb, 
    param_distributions=param_grid_nb, 
    cv=tscv, 
    scoring="f1_macro",  # mismo criterio que Random Forest y el NB propio, para poder comparar
    n_jobs=-1,
    n_iter=9,  # cubre el grid completo (9 valores de alpha)
    random_state=42
)
random_nb.fit(X_train, y_train)

y_pred_nb = random_nb.predict(X_test)

In [ ]:
#Grafico: macro-F1 (CV temporal) vs alpha (NB sklearn)
import matplotlib.pyplot as plt

resultados_nb_sklearn = pd.DataFrame(random_nb.cv_results_)[
    ["param_model__alpha", "mean_test_score", "std_test_score"]
].rename(columns={"param_model__alpha": "alpha"}).sort_values("alpha")

plt.plot(resultados_nb_sklearn["alpha"], resultados_nb_sklearn["mean_test_score"], marker="o")
plt.xscale("log")
plt.xlabel("alpha (escala log)")
plt.ylabel("macro-F1 promedio (CV temporal)")
plt.title("NB sklearn: macro-F1 vs alpha (validación cruzada temporal)")
plt.grid(True)
plt.show()

resultados_nb_sklearn

In [ ]:
#Estimador 3 (nuestro - ID3)

anios_validacion = [2019, 2020, 2021, 2022, 2023]

valores_min_info_gain = [
    0.001,
    0.005,
    0.01,
    0.02,
    0.05,
    0.1,
]

resultados_id3 = []

for min_info_gain in valores_min_info_gain:

    for anio_validacion in anios_validacion:
        mascara_entrenamiento = (
            train["date"].dt.year < anio_validacion
        )
        mascara_validacion = (
            train["date"].dt.year == anio_validacion
        )

        X_entrenamiento = train.loc[
            mascara_entrenamiento,
            atributos
        ]
        y_entrenamiento = train.loc[
            mascara_entrenamiento,
            "resultado"
        ]

        X_validacion = train.loc[
            mascara_validacion,
            atributos
        ]
        y_validacion = train.loc[
            mascara_validacion,
            "resultado"
        ]

        modelo_fold = ID3Classifier(
            criterio="Ganancia",
            min_info_gain=min_info_gain
        )

        modelo_fold.fit(
            X_entrenamiento,
            y_entrenamiento
        )

        predicciones = modelo_fold.predict(
            X_validacion
        )

        resultados_id3.append({
            "min_info_gain": min_info_gain,
            "anio_validacion": anio_validacion,
            "macro_f1": f1_score(
                y_validacion,
                predicciones,
                average="macro",
                zero_division=0
            ),
            "accuracy": accuracy_score(
                y_validacion,
                predicciones
            )
        })

In [ ]:
resultados_id3_df = pd.DataFrame(resultados_id3)

resumen_id3 = (
    resultados_id3_df
    .groupby("min_info_gain", as_index=False)
    .agg(
        macro_f1_promedio=("macro_f1", "mean"),
        macro_f1_desviacion=("macro_f1", "std"),
        accuracy_promedio=("accuracy", "mean")
    )
    .sort_values(
        "macro_f1_promedio",
        ascending=False
    )
)

resumen_id3

In [ ]:
#Estimador 4 (nuestro - Naive Bayes)
import math as math

class NaiveBayes:
    def __init__(self, m=1.0):
        self.m = m                     # float: hiperparametro del m-estimador
        self.clases = None             # list[str]: clases posibles, ej ["E", "G", "P"]
        self.atributos = None          # list[str]: nombres de los atributos
        self.prior = {}                # dict[str, float]: prior[clase] = P(clase)
        self.cantidad_por_clase = {}   # dict[str, int]: filas de cada clase (el "n" de la formula)
        self.p_attr = {}               # dict[str, float]: p_attr[col] = 1 / (valores distintos)
        self.cond = {}                 # dict: cond[col][valor][clase] = P(col = valor | clase)

    # Entrenamiento 
    def fit(self, X, y):
        # X: DataFrame, una columna por atributo
        # y: Series, la clase de cada fila ("E" / "G" / "P")

        lista_clases = sorted(y.unique())
        lista_columnas = list(X.columns)
        cantidad_filas = len(y)

        self.clases = lista_clases
        self.atributos = lista_columnas

        for clase in lista_clases:
            filas_de_la_clase = (y == clase).sum()               
            self.cantidad_por_clase[clase] = filas_de_la_clase
            self.prior[clase] = filas_de_la_clase / cantidad_filas

        for atributo in lista_columnas:
            valores_posibles = X[atributo].unique()            
            p = 1.0 / len(valores_posibles)                     
            self.p_attr[atributo] = p
            self.cond[atributo] = {}

            for valor in valores_posibles:
                self.cond[atributo][valor] = {}

                for clase in lista_clases:
                    true_si_es_de_mi_clase = (y == clase)                             # Series de True/False
                    true_si_es_de_mi_clase_y_de_mi_valor = true_si_es_de_mi_clase & (X[atributo] == valor)

                    n = true_si_es_de_mi_clase.sum()                                 
                    n_c = true_si_es_de_mi_clase_y_de_mi_valor.sum()                        

                    self.cond[atributo][valor][clase] = (n_c + self.m * p) / (n + self.m)

        return self

    def _score_fila(self, instancia, clase):
        sumaLog = 0
        for atr in self.atributos:
            valor = instancia[atr]
            if valor in self.cond[atr]:
                prob = self.cond[atr][valor][clase]
            else:
                prob = (0 + self.m * self.p_attr[atr]) / (self.cantidad_por_clase[clase] + self.m)

            sumaLog += math.log2(prob)
        res = sumaLog + math.log2(self.prior[clase])
        return res

    def predict(self, X):
        resultados = []
        for _, instancia in X.iterrows():
            scores = {
                'G': self._score_fila(instancia, 'G'),
                'E': self._score_fila(instancia, 'E'),
                'P': self._score_fila(instancia, 'P'),
            }
            resultado = max(scores, key=scores.get)   # la clave (clase) con score mas alto
            resultados.append(resultado)
        return resultados

# La instanciacion final (con el mejor m) se hace mas abajo,
# una vez que la validacion cruzada temporal elige el hiperparametro.

In [ ]:
anios_validacion = [2019, 2020, 2021, 2022, 2023]

valores_m = [
    1,
    10,
    50,
    100,
    500,
    1000,
    5000,
    10000,
    50000
]

resultados_nb = []

for m in valores_m:

    for anio_validacion in anios_validacion:
        mascara_entrenamiento = (
            train["date"].dt.year < anio_validacion
        )
        mascara_validacion = (
            train["date"].dt.year == anio_validacion
        )

        X_entrenamiento = train.loc[
            mascara_entrenamiento,
            atributos
        ]
        y_entrenamiento = train.loc[
            mascara_entrenamiento,
            "resultado"
        ]

        X_validacion = train.loc[
            mascara_validacion,
            atributos
        ]
        y_validacion = train.loc[
            mascara_validacion,
            "resultado"
        ]

        modelo_fold = NaiveBayes(m)
        modelo_fold.fit(X_entrenamiento, y_entrenamiento)

        predicciones = modelo_fold.predict(
            X_validacion
        )

        resultados_nb.append({
            "m": m,
            "anio_validacion": anio_validacion,
            "macro_f1": f1_score(
                y_validacion,
                predicciones,
                average="macro",
                zero_division=0
            ),
            "accuracy": accuracy_score(
                y_validacion,
                predicciones
            )
        })

resultados_nb_df = pd.DataFrame(resultados_nb)

resumen_nb = (
    resultados_nb_df
    .groupby("m", as_index=False)
    .agg(
        macro_f1_promedio=("macro_f1", "mean"),
        macro_f1_desviacion=("macro_f1", "std"),
        accuracy_promedio=("accuracy", "mean")
    )
    .sort_values(
        "macro_f1_promedio",
        ascending=False
    )
)

resumen_nb

In [ ]:
#Grafico: macro-F1 (CV temporal) vs m, eleccion del mejor m y entrenamiento final
import matplotlib.pyplot as plt

resumen_nb_ordenado = resumen_nb.sort_values("m")

plt.plot(resumen_nb_ordenado["m"], resumen_nb_ordenado["macro_f1_promedio"], marker="o")
plt.xscale("log")
plt.xlabel("m (escala log)")
plt.ylabel("macro-F1 promedio (CV temporal, folds por año)")
plt.title("NB nuestro: macro-F1 vs m (validación cruzada temporal)")
plt.grid(True)
plt.show()

# Mejor m segun la CV temporal 
mejor_m_nb = resumen_nb.iloc[0]["m"]
print(f"Mejor m segun CV temporal: {mejor_m_nb}")

# Se entrena el modelo final con el mejor m y se evalua una unica vez contra test 
mi_nb = NaiveBayes(m=mejor_m_nb).fit(X_train, y_train)
y_pred_mi_nb = mi_nb.predict(X_test)

In [ ]:
#Estimador 5 (Resultado mas probable (10 anos))
subset = (ds["date"].dt.year >= 2014) & (ds["date"].dt.year < 2024)
subset_l = ds.loc[subset, "resultado"]

mas_sale = subset.mode()[0]

In [ ]:
#Estadisticas
print("Mejores hiperparámetros (Random Forest):", random_rf.best_params_)
print("Accuracy (Random Forest):", accuracy_score(y_test, y_pred_rf))
print("\nReporte de clasificación: (Random Forest)\n", classification_report(y_test, y_pred_rf))

print("Mejores hiperparámetros (Naive Bayes):", random_nb.best_params_)
print("Accuracy (Naive Bayes):", accuracy_score(y_test, y_pred_nb))
print("\nReporte de clasificación: (Naive Bayes)\n", classification_report(y_test, y_pred_nb))

total = (y_test == "G").sum()/len(y_test)
print("Accuracy (NB nuestro):", accuracy_score(y_test, y_pred_mi_nb))
print("\nReporte de clasificación: (NB nuestro)\n", classification_report(y_test, y_pred_mi_nb))
print("Estimador base:", total)

In [ ]:
#Matrices de confusion en el conjunto de evaluacion
from sklearn.metrics import ConfusionMatrixDisplay

clases_orden = ["E", "G", "P"]  

candidatos = {
    "Random Forest": "y_pred_rf",
    "Naive Bayes (sklearn)": "y_pred_nb",
    "Naive Bayes (propio)": "y_pred_mi_nb",
    "ID3 (propio)": "y_pred_id3",
    "Estimador base": "y_pred_base",
}

modelos_finales = {
    nombre: globals()[var]
    for nombre, var in candidatos.items()
    if var in globals()
}

if not modelos_finales:
    print("Todavia no hay ninguna prediccion sobre test calculada.")
else:
    fig, axes = plt.subplots(1, len(modelos_finales), figsize=(5 * len(modelos_finales), 4))
    if len(modelos_finales) == 1:
        axes = [axes]  

    for ax, (nombre, y_pred) in zip(axes, modelos_finales.items()):
        ConfusionMatrixDisplay.from_predictions(
            y_test, y_pred, labels=clases_orden, ax=ax, colorbar=False
        )
        ax.set_title(nombre)

    plt.tight_layout()
    plt.show()